In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from delta.tables import DeltaTable
from datetime import datetime
import uuid
import traceback

CATALOG = "Stream_Bank_Processing"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

LOG_TABLE = f"{SILVER}.silver_processing_log"
QUARANTINE_TABLE = f"{SILVER}.silver_quarantine"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
    run_id STRING,
    batch_id STRING,
    entity_name STRING,
    source_table STRING,
    target_table STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    source_count BIGINT,
    inserted_count BIGINT,
    updated_count BIGINT,
    deleted_count BIGINT,
    duplicate_count BIGINT,
    rejected_count BIGINT,
    status STRING,
    error_message STRING
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {QUARANTINE_TABLE} (
    run_id STRING,
    batch_id STRING,
    entity_name STRING,
    rejection_reason STRING,
    rejected_at TIMESTAMP,
    record_json STRING
) USING DELTA
""")

print("Silver schema and log tables are ready")

In [0]:
def write_log(
    run_id,
    batch_id,
    entity,
    source_table,
    target_table,
    start_time,
    source_count=0,
    inserted_count=0,
    updated_count=0,
    deleted_count=0,
    duplicate_count=0,
    rejected_count=0,
    status="SUCCESS",
    error_message=None
):
    log_data = [(
        run_id,
        str(batch_id) if batch_id is not None else None,
        entity,
        source_table,
        target_table,
        start_time,
        datetime.utcnow(),
        int(source_count),
        int(inserted_count),
        int(updated_count),
        int(deleted_count),
        int(duplicate_count),
        int(rejected_count),
        status,
        error_message
    )]

    schema = T.StructType([
        T.StructField("run_id", T.StringType()),
        T.StructField("batch_id", T.StringType()),
        T.StructField("entity_name", T.StringType()),
        T.StructField("source_table", T.StringType()),
        T.StructField("target_table", T.StringType()),
        T.StructField("start_time", T.TimestampType()),
        T.StructField("end_time", T.TimestampType()),
        T.StructField("source_count", T.LongType()),
        T.StructField("inserted_count", T.LongType()),
        T.StructField("updated_count", T.LongType()),
        T.StructField("deleted_count", T.LongType()),
        T.StructField("duplicate_count", T.LongType()),
        T.StructField("rejected_count", T.LongType()),
        T.StructField("status", T.StringType()),
        T.StructField("error_message", T.StringType())
    ])

    log_df = spark.createDataFrame(log_data, schema)

    log_df.write.format("delta").mode("append").saveAsTable(LOG_TABLE)

In [0]:
def table_exists(table_name):
    return spark.catalog.tableExists(table_name)


def get_columns(df):
    return df.columns


def add_silver_metadata(df):
    return (
        df
        .withColumn("silver_processed_at", F.current_timestamp())
    )


def normalize_strings(df):
    for column_name, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(
                column_name,
                F.when(
                    F.trim(F.col(column_name)) == "",
                    F.lit(None)
                ).otherwise(F.trim(F.col(column_name)))
            )
    return df


def write_quarantine(
    df,
    run_id,
    batch_id,
    entity,
    rejection_reason
):
    if df.limit(1).count() == 0:
        return

    quarantine_df = (
        df
        .withColumn("run_id", F.lit(run_id))
        .withColumn("batch_id", F.lit(str(batch_id)))
        .withColumn("entity_name", F.lit(entity))
        .withColumn("rejection_reason", F.lit(rejection_reason))
        .withColumn("rejected_at", F.current_timestamp())
        .withColumn(
            "record_json",
            F.to_json(F.struct(*[F.col(c) for c in df.columns]))
        )
        .select(
            "run_id",
            "batch_id",
            "entity_name",
            "rejection_reason",
            "rejected_at",
            "record_json"
        )
    )

    quarantine_df.write.format("delta").mode("append").saveAsTable(
        QUARANTINE_TABLE
    )


def mask_phone(df):
    # Mask in place: keep only the last 4 digits
    if "phone" in df.columns:
        df = df.withColumn(
            "phone",
            F.when(
                F.col("phone").isNotNull(),
                F.concat(
                    F.lit("*****"),
                    F.right(F.col("phone"), F.lit(4))
                )
            )
        )
    return df


def mask_email(df):
    # Mask in place: keep first 2 chars of local part + domain
    if "email" in df.columns:
        local_part = F.split(F.col("email"), "@").getItem(0)
        domain_part = F.split(F.col("email"), "@").getItem(1)
        df = df.withColumn(
            "email",
            F.when(
                F.col("email").isNotNull()
                & F.col("email").contains("@"),
                F.concat(
                    F.left(local_part, F.lit(2)),
                    F.lit("***@"),
                    domain_part
                )
            )
        )
    return df


def hash_column(df, column_name):
    if column_name in df.columns:
        df = df.withColumn(
            f"{column_name}_hash",
            F.sha2(F.col(column_name), 256)
        )
    return df

In [0]:

def process_scd2(
    entity,
    source_table,
    target_table,
    primary_key,
    tracked_columns
):
    run_id = str(uuid.uuid4())
    start_time = datetime.utcnow()

    source_count = 0
    inserted_count = 0
    updated_count = 0
    deleted_count = 0
    duplicate_count = 0
    rejected_count = 0
    batch_id = None

    try:

        # 1. Read Bronze

        source_df = spark.table(source_table)
        source_df = source_df.cache()
        source_count = source_df.count()
        if source_count == 0:
            write_log(
                run_id,
                None,
                entity,
                source_table,
                target_table,
                start_time,
                status="SUCCESS",
                error_message="No records received"
            )

            return


        # 2. Normalize strings
        source_df = normalize_strings(source_df)

        # 3. Check required columns

        required_columns = [
            primary_key,
            "operation"
        ]

        missing_columns = [
            c for c in required_columns
            if c not in source_df.columns
        ]

        if missing_columns:

            raise ValueError(
                f"Missing required columns: {missing_columns}"
            )


        # Capture batch ID

        if "batch_id" in source_df.columns:

            batch_id = (
                source_df
                .select("batch_id")
                .first()[0]
            )


        # 4. Normalize CDC operation
        source_df = source_df.withColumn(
            "_operation",
            F.upper(
                F.trim(
                    F.col("operation")
                )
            )
        )

        source_df = source_df.withColumn(
            "_operation",

            F.when(
                F.col("_operation").isin(
                    "INSERT",
                    "I",
                    "CREATE"
                ),
                F.lit("INSERT")
            )

            .when(
                F.col("_operation").isin(
                    "UPDATE",
                    "U"
                ),
                F.lit("UPDATE")
            )

            .when(
                F.col("_operation").isin(
                    "DELETE",
                    "D"
                ),
                F.lit("DELETE")
            )

            .otherwise(
                F.lit("INVALID")
            )
        )


        # 5. Quarantine invalid operations

        invalid_df = source_df.filter(
            F.col("_operation") == "INVALID"
        )

        rejected_count = invalid_df.count()

        write_quarantine(
            invalid_df,
            run_id,
            batch_id,
            entity,
            "INVALID_OPERATION"
        )

        source_df = source_df.filter(
            F.col("_operation") != "INVALID"
        )


        # 6. Reject missing primary keys

        invalid_key_df = source_df.filter(
            F.col(primary_key).isNull()
        )

        rejected_key_count = invalid_key_df.count()

        rejected_count += rejected_key_count

        write_quarantine(
            invalid_key_df,
            run_id,
            batch_id,
            entity,
            "MISSING_PRIMARY_KEY"
        )

        source_df = source_df.filter(
            F.col(primary_key).isNotNull()
        )


        # 7. Deduplicate

        duplicate_count = (
            source_df.count()
            -
            source_df.select(
                primary_key
            ).distinct().count()
        )

        from pyspark.sql.window import Window

        if "event_timestamp" in source_df.columns:

            window = (
                Window
                .partitionBy(primary_key)
                .orderBy(
                    F.col(
                        "event_timestamp"
                    ).desc_nulls_last(),

                    F.col(
                        "ingestion_timestamp"
                    ).desc_nulls_last()
                )
            )

        else:

            window = (
                Window
                .partitionBy(primary_key)
                .orderBy(
                    F.col(
                        "ingestion_timestamp"
                    ).desc_nulls_last()
                )
            )

        source_df = (
            source_df
            .withColumn(
                "_row_number",
                F.row_number().over(window)
            )

            .filter(
                F.col("_row_number") == 1
            )

            .drop("_row_number")
        )


        # 8. Build tracked hash

        available_tracked = [
            c for c in tracked_columns
            if c in source_df.columns
        ]

        hash_values = [
            F.coalesce(
                F.col(c).cast("string"),
                F.lit("∅")
            )
            for c in available_tracked
        ]

        source_df = source_df.withColumn(
            "_record_hash",

            F.sha2(
                F.concat_ws(
                    "||",
                    *hash_values
                ),
                256
            )
        )


        # 8b. Mask PII before writing to Silver
        source_df = mask_phone(source_df)
        source_df = mask_email(source_df)

        # 9. Add SCD2 columns

        source_df = (
            source_df
            .withColumn(
                "start_date",

                F.coalesce(
                    F.col("event_timestamp"),
                    F.col("ingestion_timestamp"),
                    F.current_timestamp()
                )
            )

            .withColumn(
                "end_date",
                F.lit(None).cast("timestamp")
            )

            .withColumn(
                "is_current",
                F.lit(True)
            )

            .withColumn(
                "silver_processed_at",
                F.current_timestamp()
            )
        )



        drop_columns = [
            "operation",
            "event_timestamp",
            "_rescued_data",
            "load_type"
        ]

        for c in drop_columns:

            if c in source_df.columns:

                source_df = source_df.drop(c)


        # 11. Create initial Silver table

        if not table_exists(target_table):

            initial_df = (

                source_df

                .filter(
                    F.col("is_current") == True
                )

                # Remove CDC column before Silver write
                .drop("_operation")
            )

            initial_df.write.format(
                "delta"
            ).mode(
                "overwrite"
            ).saveAsTable(
                target_table
            )

            inserted_count = initial_df.count()

            write_log(
                run_id,
                batch_id,
                entity,
                source_table,
                target_table,
                start_time,

                source_count=source_count,
                inserted_count=inserted_count,
                duplicate_count=duplicate_count,
                rejected_count=rejected_count,

                status="SUCCESS"
            )

            return


        # 12. Read existing Silver
        target_df = spark.table(target_table)

        required_scd_columns = [
            "start_date",
            "end_date",
            "is_current",
            "_record_hash"
        ]

        missing_scd_columns = [
            c for c in required_scd_columns
            if c not in target_df.columns
        ]

        if missing_scd_columns:

            raise ValueError(
                "Target table missing SCD columns: "
                f"{missing_scd_columns}"
            )


        current_df = target_df.filter(
            F.col("is_current") == True
        ).select(
            F.col(primary_key).alias("_tgt_key"),
            F.col("_record_hash").alias("_tgt_hash")
        )


        # 13. Identify new, changed, and deleted records

        source_alias = source_df.alias("src")
        current_alias = F.broadcast(current_df).alias("tgt")

        comparison_df = (

            source_alias

            .join(
                current_alias,

                F.col(
                    f"src.{primary_key}"
                )
                ==
                F.col(
                    "tgt._tgt_key"
                ),

                "left"
            )
        )

        comparison_df = comparison_df.cache()


        # New records

        new_records = comparison_df.filter(

            F.col(
                "_tgt_key"
            ).isNull()

            &

            (
                F.col(
                    "_operation"
            ) != "DELETE"
            )
        )


        # Changed records

        changed_records = comparison_df.filter(

            F.col(
                "_tgt_key"
            ).isNotNull()

            &

            (
                F.col(
                    "_record_hash"
                )

                !=

                F.col(
                    "_tgt_hash"
                )
            )

            &

            (
                F.col(
                    "_operation"
                ) != "DELETE"
            )
        )


        # Deleted records

        delete_records = comparison_df.filter(

            F.col(
                "_tgt_key"
            ).isNotNull()

            &

            (
                F.col(
                    "_operation"
                ) == "DELETE"
            )
        )


        # 14. Expire old versions

        expire_keys = (

            changed_records

            .select(primary_key)

            .union(

                delete_records.select(primary_key)
            )

            .distinct()
        )


        # Create temporary view

        expire_keys.createOrReplaceTempView(
            "expire_keys"
        )


        # Expire current Silver versions

        DeltaTable.forName(
            spark,
            target_table
        ).alias("tgt").merge(

            expire_keys.alias("src"),

            f"tgt.{primary_key} = src.{primary_key}"
            " AND tgt.is_current = true"

        ).whenMatchedUpdate(

            set={

                "end_date": F.current_timestamp(),

                "is_current": F.lit(False)

            }

        ).execute()


        # 15. Insert new and changed versions

        insert_df = (

            new_records

            .drop("_tgt_key", "_tgt_hash")

            .unionByName(

                changed_records.drop(
                    "_tgt_key", "_tgt_hash"
                )
            )
        )


        # Remove CDC technical column before Silver write

        insert_df = insert_df.drop(
            "_operation"
        )


        # Metrics

        inserted_count = new_records.count()

        updated_count = changed_records.count()

        deleted_count = delete_records.count()


        # Append new and changed records

        if insert_df.limit(1).count() > 0:
            insert_df.coalesce(1).write.format(
                "delta"
            ).mode(
                "append"
            ).saveAsTable(
                target_table
            )


        # 16. Log success

        write_log(
            run_id,
            batch_id,
            entity,
            source_table,
            target_table,
            start_time,

            source_count=source_count,
            inserted_count=inserted_count,
            updated_count=updated_count,
            deleted_count=deleted_count,
            duplicate_count=duplicate_count,
            rejected_count=rejected_count,

            status="SUCCESS"
        )


    # 17. Error handling

    except Exception as e:

        error_message = traceback.format_exc()

        write_log(
            run_id,
            batch_id,
            entity,
            source_table,
            target_table,
            start_time,

            source_count=source_count,
            inserted_count=inserted_count,
            updated_count=updated_count,
            deleted_count=deleted_count,
            duplicate_count=duplicate_count,
            rejected_count=rejected_count,

            status="FAILED",
            error_message=error_message
        )

        raise

    finally:
        try:
            source_df.unpersist()
            comparison_df.unpersist()
        except Exception:
            pass


In [0]:
process_scd2(
    entity="customers",
    source_table=f"{BRONZE}.customers",
    target_table=f"{SILVER}.customers",
    primary_key="customer_id",
    tracked_columns=[
        "first_name",
        "last_name",
        "email",
        "phone",
        "city",
        "state"
    ]
)

process_scd2(
    entity="accounts",
    source_table=f"{BRONZE}.accounts",
    target_table=f"{SILVER}.accounts",
    primary_key="account_id",
    tracked_columns=[
        "customer_id",
        "account_type",
        "branch",
        "status",
        "balance",
        "currency"
    ]
)

In [0]:
PARTITION_MIN_ROWS = 250_000


def process_transactions():

    run_id = str(uuid.uuid4())
    start_time = datetime.utcnow()

    source_table = f"{BRONZE}.transactions"
    target_table = f"{SILVER}.transactions"

    try:

        df = spark.table(source_table)
        df = df.cache()

        source_count = df.count()

        df = normalize_strings(df)

        required_columns = [
            "txn_id",
            "account_id",
            "amount"
        ]

        missing = [
            c for c in required_columns
            if c not in df.columns
        ]

        if missing:
            raise ValueError(
                f"Missing transaction columns: {missing}"
            )

        # Validate transaction amount
        invalid_df = df.filter(
            F.col("txn_id").isNull()
            | F.col("account_id").isNull()
            | F.col("amount").isNull()
            | (F.col("amount") <= 0)
        )

        rejected_count = invalid_df.count()

        write_quarantine(
            invalid_df,
            run_id,
            None,
            "transactions",
            "INVALID_TRANSACTION"
        )

        df = df.filter(
            F.col("txn_id").isNotNull()
            & F.col("account_id").isNotNull()
            & F.col("amount").isNotNull()
            & (F.col("amount") > 0)
        )

        # Deduplication
        duplicate_count = (
            df.count()
            - df.select("txn_id").distinct().count()
        )

        df = df.dropDuplicates(["txn_id"])
        accounts = F.broadcast(
            spark.table(
                f"{SILVER}.accounts"
            ).filter(
                F.col("is_current") == True
            ).select("account_id")
        )

        invalid_accounts = df.join(
            accounts,
            "account_id",
            "left_anti"
        )

        rejected_count += invalid_accounts.count()

        write_quarantine(
            invalid_accounts,
            run_id,
            None,
            "transactions",
            "ACCOUNT_NOT_FOUND"
        )

        df = df.join(accounts, "account_id", "inner")

        # Metadata
        df = add_silver_metadata(df)
        df = df.withColumn(
            "txn_date",
            F.to_date(F.col("txn_timestamp"))
        )

        # Initial table
        if not table_exists(target_table):

            inserted_count = df.count()
            if inserted_count >= PARTITION_MIN_ROWS:
                (
                    df.repartition("txn_date")
                    .write.format("delta")
                    .partitionBy("txn_date")
                    .mode("overwrite")
                    .saveAsTable(target_table)
                )
            else:
                df.coalesce(1).write.format("delta").mode(
                    "overwrite"
                ).saveAsTable(target_table)

        else:

            existing = spark.table(target_table)

            new_df = df.join(
                existing.select("txn_id"),
                "txn_id",
                "left_anti"
            )

            inserted_count = new_df.count()

            if inserted_count > 0:
                new_df.coalesce(1).write.format("delta").mode(
                    "append"
                ).saveAsTable(target_table)

        write_log(
            run_id,
            None,
            "transactions",
            source_table,
            target_table,
            start_time,
            source_count=source_count,
            inserted_count=inserted_count,
            duplicate_count=duplicate_count,
            rejected_count=rejected_count,
            status="SUCCESS"
        )

    except Exception:

        write_log(
            run_id,
            None,
            "transactions",
            source_table,
            target_table,
            start_time,
            status="FAILED",
            error_message=traceback.format_exc()
        )

        raise

    finally:

        # OPTIMIZATION (housekeeping): release the cached batch.
        try:
            df.unpersist()
        except Exception:
            pass

process_transactions()

In [0]:
%sql
SELECT *
FROM Stream_Bank_Processing.silver.silver_processing_log
ORDER BY start_time DESC;

In [0]:
%sql
SELECT *
FROM Stream_Bank_Processing.silver.customers
-- Where customer_id = 'C00052'
-- WHERE is_current = FALSE;

In [0]:
%sql
select * from stream_bank_processing.silver.customers
-- where is_current = FALSE
-- where account_id = 'A001026'

In [0]:
target_table = f"{SILVER}.customers"

if table_exists(target_table):

    dt = DeltaTable.forName(spark, target_table)

    dt.update(
        condition=F.col("phone").isNotNull(),
        set={
            "phone": F.concat(
                F.lit("*****"),
                F.right(F.col("phone"), F.lit(4))
            )
        }
    )

    dt.update(
        condition=(
            F.col("email").isNotNull()
            & F.col("email").contains("@")
        ),
        set={
            "email": F.concat(
                F.left(
                    F.split(F.col("email"), "@").getItem(0),
                    F.lit(2)
                ),
                F.lit("***@"),
                F.split(F.col("email"), "@").getItem(1)
            )
        }
    )

    print("Backfill complete")

    display(
        spark.table(target_table).select(
            "customer_id", "phone", "email"
        ).limit(10)
    )

else:
    print(f"{target_table} does not exist yet")

In [0]:
%sql
Select * from stream_bank_processing.silver.silver_processing_log
order by end_time desc

In [0]:
%sql
